In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

delhi_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase2_audit.parquet"
    )
)

delhi_df["Datetime"] = pd.to_datetime(
    delhi_df["Datetime"]
)

delhi_df = delhi_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Records:", len(delhi_df))
print("Stations:", delhi_df["StationId"].nunique())

In [ ]:
target = "AQI"

key_pollutants = [
    "PM2.5",
    "PM10",
    "NO2",
    "NOx",
    "CO",
    "SO2",
    "O3"
]

print("Target:", target)
print("Key pollutants:", key_pollutants)

In [ ]:
station_basic = (
    delhi_df
    .groupby(
        ["StationId", "StationName", "City", "State"],
        dropna=False
    )
    .agg(
        TotalRecords=("Datetime", "size"),
        FirstObservation=("Datetime", "min"),
        LastObservation=("Datetime", "max"),
        ValidAQI=("AQI", "count")
    )
    .reset_index()
)

station_basic["AQIMissingPct"] = (
    1
    - station_basic["ValidAQI"]
    / station_basic["TotalRecords"]
) * 100

station_basic["CoverageDays"] = (
    (
        station_basic["LastObservation"]
        - station_basic["FirstObservation"]
    )
    .dt.total_seconds()
    / 86400
)

display(
    station_basic.sort_values(
        "TotalRecords",
        ascending=False
    )
)

In [ ]:
pollutant_missing = (
    delhi_df
    .groupby("StationId")[
        key_pollutants
    ]
    .apply(
        lambda x: x.isna().mean() * 100
    )
    .reset_index()
)

pollutant_missing["MeanPollutantMissingPct"] = (
    pollutant_missing[
        key_pollutants
    ]
    .mean(axis=1)
)

pollutant_missing["MaxPollutantMissingPct"] = (
    pollutant_missing[
        key_pollutants
    ]
    .max(axis=1)
)

display(
    pollutant_missing.sort_values(
        "MeanPollutantMissingPct"
    )
)

In [ ]:
station_quality = station_basic.merge(
    pollutant_missing[
        [
            "StationId",
            "MeanPollutantMissingPct",
            "MaxPollutantMissingPct"
        ]
    ],
    on="StationId",
    how="left",
    validate="one_to_one"
)

display(
    station_quality.sort_values(
        [
            "AQIMissingPct",
            "MeanPollutantMissingPct"
        ]
    )
)

In [ ]:
continuity_rows = []

for station_id, group in delhi_df.groupby(
    "StationId"
):

    times = (
        group["Datetime"]
        .drop_duplicates()
        .sort_values()
    )

    if len(times) <= 1:
        continuity = 0.0
    else:
        diffs = times.diff().dropna()

        continuity = (
            diffs.eq(
                pd.Timedelta(hours=1)
            ).mean()
            * 100
        )

    continuity_rows.append(
        {
            "StationId": station_id,
            "HourlyContinuityPct": continuity
        }
    )

continuity_df = pd.DataFrame(
    continuity_rows
)

station_quality = station_quality.merge(
    continuity_df,
    on="StationId",
    how="left",
    validate="one_to_one"
)

display(
    station_quality.sort_values(
        "HourlyContinuityPct",
        ascending=False
    )
)

In [ ]:
aqi_continuity_rows = []

for station_id, group in delhi_df.groupby(
    "StationId"
):

    group = (
        group[
            [
                "Datetime",
                "AQI"
            ]
        ]
        .drop_duplicates("Datetime")
        .sort_values("Datetime")
    )

    valid_aqi = group["AQI"].notna()

    valid_pairs = (
        valid_aqi
        & valid_aqi.shift(1, fill_value=False)
    )

    timestamp_pairs = (
        group["Datetime"].diff()
        == pd.Timedelta(hours=1)
    )

    usable_pairs = (
        valid_pairs
        & timestamp_pairs
    )

    if len(group) <= 1:
        continuity = 0.0
    else:
        continuity = (
            usable_pairs.sum()
            / max(
                len(group) - 1,
                1
            )
        ) * 100

    aqi_continuity_rows.append(
        {
            "StationId": station_id,
            "AQIHourlyContinuityPct": continuity
        }
    )

aqi_continuity_df = pd.DataFrame(
    aqi_continuity_rows
)

station_quality = station_quality.merge(
    aqi_continuity_df,
    on="StationId",
    how="left",
    validate="one_to_one"
)

display(
    station_quality.sort_values(
        "AQIHourlyContinuityPct",
        ascending=False
    )
)

In [ ]:
def minmax_score(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            1.0,
            index=series.index
        )

    return (
        (series - minimum)
        / (maximum - minimum)
    )

station_quality["RecordScore"] = minmax_score(
    station_quality["TotalRecords"]
)

station_quality["AQIAvailabilityScore"] = (
    1
    - station_quality["AQIMissingPct"] / 100
)

station_quality["PollutantCompletenessScore"] = (
    1
    - station_quality[
        "MeanPollutantMissingPct"
    ] / 100
)

station_quality["CoverageScore"] = minmax_score(
    station_quality["CoverageDays"]
)

station_quality["ContinuityScore"] = (
    station_quality[
        "HourlyContinuityPct"
    ] / 100
)

station_quality["AQIContinuityScore"] = (
    station_quality[
        "AQIHourlyContinuityPct"
    ] / 100
)

display(
    station_quality[
        [
            "StationId",
            "RecordScore",
            "AQIAvailabilityScore",
            "PollutantCompletenessScore",
            "CoverageScore",
            "ContinuityScore",
            "AQIContinuityScore"
        ]
    ]
)

In [ ]:
station_quality["OverallQualityScore"] = (
    0.20
    * station_quality["RecordScore"]
    +
    0.25
    * station_quality["AQIAvailabilityScore"]
    +
    0.20
    * station_quality["PollutantCompletenessScore"]
    +
    0.10
    * station_quality["CoverageScore"]
    +
    0.10
    * station_quality["ContinuityScore"]
    +
    0.15
    * station_quality["AQIContinuityScore"]
)

station_quality = station_quality.sort_values(
    "OverallQualityScore",
    ascending=False
).reset_index(drop=True)

display(
    station_quality[
        [
            "StationId",
            "StationName",
            "TotalRecords",
            "AQIMissingPct",
            "MeanPollutantMissingPct",
            "CoverageDays",
            "HourlyContinuityPct",
            "AQIHourlyContinuityPct",
            "OverallQualityScore"
        ]
    ]
)

In [ ]:
top_quality_candidates = station_quality.head(
    20
).copy()

display(
    top_quality_candidates[
        [
            "StationId",
            "StationName",
            "TotalRecords",
            "ValidAQI",
            "AQIMissingPct",
            "MeanPollutantMissingPct",
            "CoverageDays",
            "HourlyContinuityPct",
            "AQIHourlyContinuityPct",
            "OverallQualityScore"
        ]
    ]
)

In [ ]:
station_quality["Agency"] = (
    station_quality["StationName"]
    .astype(str)
    .str.extract(
        r"-\s*(DPCC|CPCB|IMD)\s*$",
        expand=False
    )
    .fillna("Other")
)

print(
    station_quality["Agency"].value_counts()
)

In [ ]:
agency_targets = {
    "CPCB": 3,
    "DPCC": 5,
    "IMD": 2
}

selected_station_ids = []

remaining = station_quality.copy()

for agency, target_count in agency_targets.items():

    agency_candidates = (
        remaining[
            remaining["Agency"] == agency
        ]
        .sort_values(
            "OverallQualityScore",
            ascending=False
        )
        .head(target_count)
    )

    selected_station_ids.extend(
        agency_candidates["StationId"].tolist()
    )

# If fewer than 10 were selected because of availability,
# fill remaining slots using highest-quality stations.
if len(selected_station_ids) < 10:

    remaining_candidates = (
        station_quality[
            ~station_quality["StationId"]
            .isin(selected_station_ids)
        ]
        .sort_values(
            "OverallQualityScore",
            ascending=False
        )
    )

    additional = remaining_candidates.head(
        10 - len(selected_station_ids)
    )["StationId"].tolist()

    selected_station_ids.extend(
        additional
    )

selected_station_ids = selected_station_ids[
    :10
]

print(
    "Initial 10-station selection:",
    selected_station_ids
)

In [ ]:
selected_stations = (
    station_quality[
        station_quality["StationId"].isin(
            selected_station_ids
        )
    ]
    .sort_values(
        "OverallQualityScore",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    selected_stations[
        [
            "StationId",
            "StationName",
            "Agency",
            "TotalRecords",
            "ValidAQI",
            "AQIMissingPct",
            "MeanPollutantMissingPct",
            "CoverageDays",
            "AQIHourlyContinuityPct",
            "OverallQualityScore"
        ]
    ]
)

In [ ]:
print(
    "Selected stations:",
    len(selected_station_ids)
)

assert len(selected_station_ids) == 10, (
    "Exactly 10 stations must be selected."
)

In [ ]:
delhi_10_df = delhi_df[
    delhi_df["StationId"].isin(
        selected_station_ids
    )
].copy()

delhi_10_df = delhi_10_df.sort_values(
    [
        "StationId",
        "Datetime"
    ]
).reset_index(drop=True)

print(
    "10-station dataset:",
    delhi_10_df.shape
)

print(
    "Stations:",
    delhi_10_df["StationId"].nunique()
)

In [ ]:
station_distribution = (
    delhi_10_df
    .groupby(
        ["StationId", "StationName"]
    )
    .size()
    .sort_values(
        ascending=False
    )
    .to_frame("Records")
)

display(
    station_distribution
)

In [ ]:
aqi_availability_10 = (
    delhi_10_df
    .groupby(
        ["StationId", "StationName"]
    )["AQI"]
    .agg(
        TotalRecords="size",
        ValidAQI="count"
    )
)

aqi_availability_10["AQIMissingPct"] = (
    1
    -
    aqi_availability_10["ValidAQI"]
    /
    aqi_availability_10["TotalRecords"]
) * 100

display(
    aqi_availability_10.sort_values(
        "AQIMissingPct"
    )
)

In [ ]:
ten_station_dataset_path = os.path.join(
    PROCESSED_PATH,
    "delhi_10_station_raw.parquet"
)

delhi_10_df.to_parquet(
    ten_station_dataset_path,
    index=False
)

print(
    "10-station dataset saved:"
)

print(
    ten_station_dataset_path
)

In [ ]:
station_quality.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_75_all_station_quality_scores.csv"
    ),
    index=False
)

selected_stations.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_75_selected_10_stations.csv"
    ),
    index=False
)

print("Station-selection tables saved.")

In [ ]:
selection_config = {
    "selected_station_count": 10,

    "selection_objective": (
        "Select high-quality, temporally reliable, "
        "AQI-complete and pollutant-complete Delhi stations "
        "while preserving monitoring-agency diversity."
    ),

    "score_weights": {
        "record_count": 0.20,
        "aqi_availability": 0.25,
        "pollutant_completeness": 0.20,
        "coverage_duration": 0.10,
        "hourly_continuity": 0.10,
        "aqi_hourly_continuity": 0.15
    },

    "agency_targets": agency_targets,

    "selected_station_ids": [
        str(station_id)
        for station_id in selected_station_ids
    ]
}

selection_config_path = os.path.join(
    OUTPUT_PATH,
    "phase11_75_station_selection_config.json"
)

with open(
    selection_config_path,
    "w"
) as f:
    import json

    json.dump(
        selection_config,
        f,
        indent=4
    )

print(
    "Selection configuration saved:"
)

print(
    selection_config_path
)

In [ ]:
print("=" * 70)
print("PHASE 11.75 — FINAL 10-STATION SELECTION")
print("=" * 70)

print(
    "Selected stations:",
    len(selected_station_ids)
)

display(
    selected_stations[
        [
            "StationId",
            "StationName",
            "Agency",
            "TotalRecords",
            "AQIMissingPct",
            "MeanPollutantMissingPct",
            "AQIHourlyContinuityPct",
            "OverallQualityScore"
        ]
    ]
)

print(
    "\n10-station records:",
    len(delhi_10_df)
)

print(
    "\nDataset saved:",
    ten_station_dataset_path
)

print("=" * 70)

In [ ]:
print("=" * 70)
print("PHASE 11.75 — FINAL 10-STATION SELECTION")
print("=" * 70)

print(
    "Selected stations:",
    len(selected_station_ids)
)

display(
    selected_stations[
        [
            "StationId",
            "StationName",
            "Agency",
            "TotalRecords",
            "AQIMissingPct",
            "MeanPollutantMissingPct",
            "AQIHourlyContinuityPct",
            "OverallQualityScore"
        ]
    ]
)

print(
    "\n10-station records:",
    len(delhi_10_df)
)

print(
    "\nDataset saved:",
    ten_station_dataset_path
)

print("=" * 70)

In [ ]:
session_status = {
    "completed_phase": "Phase 11.75 - Final 10-Station Selection",
    "next_phase": "Phase 12.0 - 10-Station Training/Inference Parity Audit",

    "station_count": 10,

    "selected_station_ids": [
        str(station_id)
        for station_id in selected_station_ids
    ],

    "dataset": "delhi_10_station_raw.parquet"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

print(
    "Phase 11.75 status saved."
)

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save final 10-station dataset
# --------------------------------------------------

ten_station_dataset_path = os.path.join(
    PROCESSED_PATH,
    "delhi_10_station_raw.parquet"
)

delhi_10_df.to_parquet(
    ten_station_dataset_path,
    index=False
)

# --------------------------------------------------
# 2. Save all station quality scores
# --------------------------------------------------

station_quality.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_75_all_station_quality_scores.csv"
    ),
    index=False
)

# --------------------------------------------------
# 3. Save selected 10 stations
# --------------------------------------------------

selected_stations.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase11_75_selected_10_stations.csv"
    ),
    index=False
)

# --------------------------------------------------
# 4. Save station selection configuration
# --------------------------------------------------

selection_config = {
    "selected_station_count": 10,
    "selected_station_ids": [
        str(station_id)
        for station_id in selected_station_ids
    ],
    "score_weights": {
        "record_count": 0.20,
        "aqi_availability": 0.25,
        "pollutant_completeness": 0.20,
        "coverage_duration": 0.10,
        "hourly_continuity": 0.10,
        "aqi_hourly_continuity": 0.15
    },
    "agency_targets": agency_targets,
    "dataset": "delhi_10_station_raw.parquet"
}

selection_config_path = os.path.join(
    OUTPUT_PATH,
    "phase11_75_station_selection_config.json"
)

with open(
    selection_config_path,
    "w"
) as f:
    json.dump(
        selection_config,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Save Phase 11.75 summary
# --------------------------------------------------

phase11_75_summary = {
    "completed_phase": "Phase 11.75 - Final 10-Station Selection",
    "next_phase": "Phase 12.0 - 10-Station Training/Inference Parity Audit",

    "station_count": 10,
    "records": int(len(delhi_10_df)),

    "selected_station_ids": [
        str(station_id)
        for station_id in selected_station_ids
    ]
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase11_75_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase11_75_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 6. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 11.75 - Final 10-Station Selection",
    "next_phase": "Phase 12.0 - 10-Station Training/Inference Parity Audit",
    "station_count": 10,
    "dataset": "delhi_10_station_raw.parquet"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 7. Verify
# --------------------------------------------------

files_to_check = [
    ten_station_dataset_path,
    os.path.join(
        TABLE_PATH,
        "phase11_75_all_station_quality_scores.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase11_75_selected_10_stations.csv"
    ),
    selection_config_path,
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 11.75 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nFinal station count:", 10)
print("Final records:", len(delhi_10_df))
print("Next: Phase 12.0 - 10-Station Training/Inference Parity Audit")